# Homework 2

# Set up

## Installing packages

In [ ]:
!pip install requests PyPDF2 gdown
!pip install 'markitdown[pdf]'
!pip install langchain_mcp_adapters langchain_google_genai langchain-openai

## Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `VERTEX_API_KEY`.


1.   Look for the key icon on the left panel of your colab.
2.   Under `Name`, create `VERTEX_API_KEY`.
3. Copy your key to `Value`.

If you cannot use VERTEX_API_KEY, you can use deepseek models via `DEEPSEEK_API_KEY`. It does not affect your score.



In [ ]:
from google.colab import userdata
GEMINI_VERTEX_API_KEY = userdata.get('GEMINI_API_KEY')
# DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

# Download sample CVs

## Downloading sample_cv.pdf
The codes below download the sample CV


In [ ]:
import os
import gdown

folder_id = "1adYKq7gSSczFP3iikfA8Er-HSZP6VM7D"
folder_url = f"https://drive.google.com/drive/folders/{folder_id}"

output_dir = "downloaded_cvs"
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=folder_url,
    output=output_dir,
    quiet=False,
    use_cookies=False
)

Retrieving folder contents


Processing file 1NR1RUKx4GyM7QOBxKXkfh4e8jUkxFCsp CV_1.pdf
Processing file 16lrd-uO8AAnCnv7UG9Rs_Nk6SUu0Iwbs CV_2.pdf
Processing file 15hVEuBan_EKhEty2aZBd6rcpDpP4o7Vr CV_3.pdf
Processing file 1Y2w_mAUEhg4vZBdvvR-0n3Jf2mKuGDRk CV_4.pdf
Processing file 1PLwkva-pdua6ZVvmLg9mxHeljq9D8C_C CV_5.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1NR1RUKx4GyM7QOBxKXkfh4e8jUkxFCsp
To: /content/downloaded_cvs/CV_1.pdf
100%|██████████| 147k/147k [00:00<00:00, 7.30MB/s]
Downloading...
From: https://drive.google.com/uc?id=16lrd-uO8AAnCnv7UG9Rs_Nk6SUu0Iwbs
To: /content/downloaded_cvs/CV_2.pdf
100%|██████████| 75.1k/75.1k [00:00<00:00, 4.89MB/s]
Downloading...
From: https://drive.google.com/uc?id=15hVEuBan_EKhEty2aZBd6rcpDpP4o7Vr
To: /content/downloaded_cvs/CV_3.pdf
100%|██████████| 72.0k/72.0k [00:00<00:00, 6.73MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Y2w_mAUEhg4vZBdvvR-0n3Jf2mKuGDRk
To: /content/downloaded_cvs/CV_4.pdf
100%|██████████| 73.3k/73.3k [00:00<00:00, 5.83MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PLwkva-pdua6ZVvmLg9mxHeljq9D8C_C
To: /content/downloaded_cvs/CV_5.pdf
100%|██████████| 97.9k/97.9k [00:00<00:00, 3.73MB/s]
Download complete

['downloaded_cvs/CV_1.pdf',
 'downloaded_cvs/CV_2.pdf',
 'downloaded_cvs/CV_3.pdf',
 'downloaded_cvs/CV_4.pdf',
 'downloaded_cvs/CV_5.pdf']

In [ ]:
# =====================================================
#  Load and display all CV PDFs in order
# =====================================================
import os
from markitdown import MarkItDown

cv_dir = "downloaded_cvs"

# Initialize MarkItDown
md = MarkItDown(enable_plugins=False)

# Collect and sort PDFs numerically
pdf_files = sorted(
    [f for f in os.listdir(cv_dir) if f.lower().endswith(".pdf")],
    key=lambda x: int("".join(filter(str.isdigit, x)))  # CV_1.pdf → 1
)

all_cvs = []

for pdf_name in pdf_files:
    pdf_path = os.path.join(cv_dir, pdf_name)
    result = md.convert(pdf_path)

    all_cvs.append({
        "file": pdf_name,
        "text": result.text_content
    })

    print("=" * 80)
    print(f"📄 {pdf_name}")
    print("=" * 80)
    print(result.text_content)
    print("\n\n")


📄 CV_1.pdf
|     |     |     |     | John         |           | Smith        |                   |     |     |
| --- | --- | --- | --- | ------------ | --------- | ------------ | ----------------- | --- | --- |
|     |     |     |     | Marketing    |           | Professional |                   |     |     |
|     |     |     |     | + Singapore, | Singapore |              | (cid:209) Kowloon |     |     |
Experience
|                |                  |     |          |                     |              |            |     | 2020 – | Present |
| -------------- | ---------------- | --- | -------- | ------------------- | ------------ | ---------- | --- | ------ | ------- |
| Engineer,      | ByteDance        |     |          |                     |              |            |     |        |         |
| • Worked       | in a fast-paced, |     | global   | technology          | environment. |            |     |        |         |
| • Collaborated | across           |     | teams to | sup

# Connect to our MCP server

Documentation about MCP: https://modelcontextprotocol.io/docs/getting-started/intro.

Using MCP servers in Langchain https://docs.langchain.com/oss/python/langchain/mcp.

## Check which tools that the MCP server provide

In [ ]:
import asyncio
import json
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})

mcp_tools = await client.get_tools()
for tool in mcp_tools:
    print(tool.name)
    print(tool.description)
    print(tool.args)
    print("\n\n------------------------------------------------------\n\n")

search_facebook_users
Search for Facebook users by display name (supports partial and fuzzy matching).

Args:
    q: Search query string (case-insensitive, matches any part of display name)
       Examples: "John", "john smith", "Smith"
    limit: Maximum number of results to return (default: 20, max: 20)
    fuzzy: Enable fuzzy matching if exact search returns no results (default: True)

Returns:
    List of user dictionaries, each containing:
    - id (int): Unique Facebook user ID for use with get_facebook_profile()
    - display_name (str): User's Facebook display name (may differ from legal name)
    - city (str): Current city of residence
    - country (str): Country of residence
    - match_type (str): "exact" or "fuzzy" (indicates search method used)
    
    Returns empty list [] if no matches found.

Example:
    search_facebook_users("Alex Chan", limit=5)
    → [{"id": 123, "display_name": "Alex Chan", "city": "Hong Kong", "country": "Hong Kong", "match_type": "exact"}]
    

## A simple agent using tools from the MCP server


In [ ]:
import os
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mcp_adapters.client import MultiServerMCPClient

# ---------------------------
# 1. Define a local tool
# ---------------------------
@tool
def say_hello(name: str) -> str:
    """Say hello to a person by name."""
    return f"Hello, {name}! 👋"

# ---------------------------
# 2. Load MCP tools + merge
# ---------------------------
client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})

mcp_tools = await client.get_tools()
tools = mcp_tools + [say_hello]

# ---------------------------
# 3. Initialize Gemini (tool-enabled) or deepseek
# ---------------------------
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_VERTEX_API_KEY,
    temperature=0,
)

# from langchain_openai import ChatOpenAI
# DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
# llm = ChatOpenAI(
#     model="deepseek-chat",          # or "deepseek-reasoner"
#     api_key=DEEPSEEK_API_KEY,
#     base_url="https://api.deepseek.com/v1",
#     temperature=0,
# )

llm_with_tools = llm.bind_tools(tools)

# ---------------------------
# 4. Single-step invocation
# ---------------------------
query = "Say hello to Bao using tool, then search for someone named Alice on Facebook."

response = llm_with_tools.invoke([
    HumanMessage(content=query)
])

print(response)

content='' additional_kwargs={'function_call': {'name': 'search_facebook_users', 'arguments': '{"q": "Alice"}'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019c9e85-efc5-7713-9d71-0399420986cc-0' tool_calls=[{'name': 'say_hello', 'args': {'name': 'Bao'}, 'id': '854909d1-59b8-4ec1-a779-32d5f4148393', 'type': 'tool_call'}, {'name': 'search_facebook_users', 'args': {'q': 'Alice'}, 'id': 'd03de4cb-a26a-47ff-b522-d3b99f4e239f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 2852, 'output_tokens': 33, 'total_tokens': 2885, 'input_token_details': {'cache_read': 0}}


In [ ]:
def information_extraction(cv:str):
  llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_VERTEX_API_KEY,
    vertexai=False,
    temperature=0
  )


  extract_information_prompt = (f"""You are a professional resume parser. Your task is to meticulously extract all relevant professional and academic information from the provided resume text and structure it into a strict JSON object.

**Strict Output Constraints:**
Your response **must** be a single JSON object string. You are **strictly forbidden** from including any explanations, conversational text, Markdown formatting (e.g., ` ```json `), or any other extraneous characters. Ensure the output is a valid, parseable JSON string.

**Extraction Rules and Missing Data Handling:**
- If a field is not found in the resume, set its value to `null` (for single string/integer/boolean values).
- If a list (e.g., `skills`, `experience`, `education`) contains no entries, it **must** be an empty list `[]`. Do not include objects with null values in these lists if no data is found.
- **name:** Extract the full name of the resume owner.
- **headline:** Identify a concise professional headline or summary, typically found near the name. Set to `null` if not found.
- **city (List[str]):** Extract a list of cities and/or countries where the person has lived, worked, or is from. Prioritize locations associated with contact information, current residence, or work history. **Crucially, do NOT include cities or countries that are solely part of an educational institution's name (e.g., "Tokyo" from "University of Tokyo"), unless they are also explicitly stated as a separate personal location.** Set to an empty list `[]` if no locations are found.
- **country:** Extract the current living country. Set to `null` if not found.
- **industry:** Infer the primary industry from headline titles. Examples: "Software", "Finance", "AI", "Consulting"


- **skills (List[dict]):** Extract a list of skills.
    - `name` (str): Skill name (e.g., "Python", "Machine Learning").

- **experience (List[dict]):** Extract work history entries.
    - `company` (str): Employer name.
    - `title` (str): Job title.
    - `start_year` (int): Employment start year.
    - `end_year` (int|None): Employment end year. Set to `null` if currently employed there.
    - `is_current` (bool): `true` if currently employed here; `false` otherwise.

- **education (List[dict]):** Extract academic history entries.
    - `school` (str): Institution name.
    - `degree` (str): Degree type (e.g., BSc, MSc, MBA, PhD). Set to `null` if not explicitly stated.
    - `field` (str): Field of study. Set to `null` if not found.
    - `start_year` (int|None): Education start year. Set to `null` if only one year is found and it's inferred as the end year.
    - `end_year` (int): Graduation year. **Crucial: If only one year is present for an education entry, assume it is the `end_year`, and set `start_year` to `null`.**

**Target JSON Structure Example (using null for missing single values and empty lists for missing arrays):**
```json
{{
    "name": "John Doe",
    "headline": "Experienced Software Engineer",
    "city": "London",
    "industry": "Technology",
    "skills": [
        {{"name": "Python"}},
        {{"name": "Machine Learning}}
    ],
    "experience": [
        {{
            "company": "Tech Corp",
            "title": "Senior Software Engineer",
            "start_year": 2018,
            "end_year": null,
            "is_current": true
        }},
        {{
            "company": "Innovate Ltd.",
            "title": "Software Developer",
            "start_year": 2015,
            "end_year": 2018,
            "is_current": false
        }}
    ],
    "education": [
        {{
            "school": "University of London",
            "degree": "MSc",
            "field": "Computer Science",
            "start_year": 2017,
            "end_year": 2018
        }},
        {{
            "school": "Local College",
            "degree": "BSc",
            "field": "Software Engineering",
            "start_year": null,
            "end_year": 2015
        }}
    ]
}}

**Input Resume Text:**
{cv}""")
  extract_information = llm.invoke(extract_information_prompt)
  extract_information = json.loads(extract_information.content)
  return extract_information

In [ ]:
first_info = information_extraction(all_cvs[0]["text"])
print(first_info)

{'name': 'John Smith', 'headline': 'Marketing Professional', 'city': ['Singapore', 'Kowloon'], 'country': 'Singapore', 'industry': 'Marketing', 'skills': [{'name': 'Content Creation'}, {'name': 'SEO'}, {'name': 'Social Media'}], 'experience': [{'company': 'ByteDance', 'title': 'Engineer', 'start_year': 2020, 'end_year': None, 'is_current': True}], 'education': [{'school': 'McGill University', 'degree': 'BSc', 'field': 'Marketing', 'start_year': None, 'end_year': 2009}]}


In [ ]:
def first_rating(extracted_info:list):
  llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_VERTEX_API_KEY,
    vertexai=False,
    temperature=0
  )
  first_rating_prompt = (f"""You are a professional resume verifier. Your task is to analyze the provided structured resume data (in JSON format) for internal inconsistencies and assign a score based on these findings.

**Inconsistency Categories to Check:**
1.  **Overlapping work experience dates:** Check if any work experiences have dates that conflict (e.g., end_year of one experience is after start_year of another, or multiple current positions).
2.  **Starting work before graduation:** Verify if any work experience started before the `end_year` of the latest relevant education. Consider only education entries with valid `end_year`.
3.  **Significant disconnect between education and listed skills:** Evaluate if the skills listed are generally irrelevant or appear too advanced/basic for the person's educational background (`education` and `skills` fields).
4.  **Significant disconnect between work experience and listed skills:** Evaluate if the skills listed are generally irrelevant or appear too advanced/basic for the person's work history (`experience` and `skills` fields).

**Scoring Logic:**
Start with a perfect score of 4 points. Deduct 1 point for each distinct type of inconsistency found. The score should not go below 0.

**Input Resume Data (JSON):**
{json.dumps(extracted_info, indent=2)}

**Output Format Requirements:**
Your response **must** be a string strictly adhering to the following JSON structure. It should contain a 'discrepancy' field describing ALL identified inconsistencies (or "No inconsistencies found.") and a 'score' field.

**Strict Output Constraints:**
Your response **must and can only** be this JSON object string. You are **strictly forbidden** from including any explanations, conversational text, Markdown formatting (e.g., ` ```json `), or any other extraneous characters.

**JSON Structure Examples:**
{{"discrepancy": "Found overlapping work experience dates and a disconnect between education and skills.", "score": 2}}
or
{{"discrepancy": "No inconsistencies found.", "score": 4}}
""")

  rating_response = llm.invoke(first_rating_prompt)
  rating_result = json.loads(rating_response.content)
  rating_result["score"] = rating_result["score"]/4
  return rating_result


In [ ]:
print(first_rating(first_info))

{'discrepancy': 'Significant disconnect between work experience and listed skills.', 'score': 0.75}


In [ ]:
async def seek_for_Facebook_id(extracted_info:list):
    client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})
    tools = await client.get_tools()
    account_list_json = await tools[0].ainvoke({"q": extracted_info["name"], "limit": 20,"fuzzy": True})
    account_list = json.loads(account_list_json[0]["text"])
    possible_account=[]
    for i in range(len(account_list)):
      if (account_list[i]["city"] in extracted_info["city"] or
          account_list[i]["country"] in extracted_info["city"]):
        possible_account.append(account_list[i]["id"])

    return possible_account

In [ ]:
await seek_for_Facebook_id(first_info)

[8, 213, 335, 377, 441, 489, 534]

In [ ]:
async def seek_for_linkedin_id(extracted_info:list):
    llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_VERTEX_API_KEY,
    vertexai=False,
    temperature=0
  )
    client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})
    tools = await client.get_tools()
    id_list = []
    for i in range(len(extracted_info["city"])):

      city_account_list_json = await tools[3].ainvoke({"q": extracted_info["name"], "limit": 10, "location": extracted_info["city"][i],"fuzzy": True})
      city_account_list = json.loads(city_account_list_json[0]["text"])

      industry_keyword = extracted_info["industry"]
      prompt_for_industry_match = (f"""You are a professional data filter. Your task is to identify and extract the 'id' of individuals whose 'industry' field is similar to a given keyword from a list of person profiles.
**Input:**
- A JSON array of dictionaries, where each dictionary represents a person's profile and includes an 'id' and an 'industry' field.
- A string `keyword` which will be used to find similar industries.

**Logic for Similarity:**
1.  You must perform a case-insensitive comparison.
2.  An industry is considered 'similar' if the provided keyword represents a related or relevant professional domain to the person's 'industry' field

**Steps:**
1.  Iterate through each person's profile (dictionary) in the provided list.
2.  For each profile, convert both the `keyword` and the profile's `industry` value to lowercase.
3.  Check if the lowercase `keyword` is present anywhere within the lowercase `industry` value.
4.  If it is, extract the `id` from that profile.
5.  Collect all such extracted `id` values into a single list.

**Output Format Requirements:**
Your response **must** be a string strictly adhering to a JSON array of integers. This array should contain only the 'id' values that match the similarity criteria.

**Strict Output Constraints:**
Your response **must and can only** be this JSON array string. You are **strictly forbidden** from including any explanations, conversational text, Markdown formatting (e.g., ` ```json `), or any other extraneous characters.

**Example Input and Output:**
Input Data:
```json
{json.dumps(city_account_list, indent=4)}
keyword: {industry_keyword}
Expected Output :
[456, 101]
""")

      remain_id = llm.invoke(prompt_for_industry_match)
      matching_ids = json.loads(remain_id.content)
      if remain_id == []:
        matching_ids =city_account_list
      id_list = id_list+matching_ids
    return id_list

In [ ]:
print(first_info)

{'name': 'John Smith', 'headline': 'Marketing Professional', 'city': ['Singapore', 'Kowloon'], 'country': 'Singapore', 'industry': 'Technology', 'skills': [{'name': 'Content Creation'}, {'name': 'SEO'}, {'name': 'Social Media'}], 'experience': [{'company': 'ByteDance', 'title': 'Engineer', 'start_year': 2020, 'end_year': None, 'is_current': True}], 'education': [{'school': 'McGill University', 'degree': 'BSc', 'field': 'Marketing', 'start_year': None, 'end_year': 2009}]}


In [ ]:
await seek_for_linkedin_id(first_info)

[9, 377]

In [ ]:
async def rate_media_info(extracted_info:list,media_type:str,id_list:list):

  llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_VERTEX_API_KEY,
    vertexai=False,
    temperature=0
  )
  client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})
  tools = await client.get_tools()
  if media_type == "Facebook":
    full_point = 3
    tool = tools[1]
    id_name = "user_id"
  elif media_type == "Linkedin":
    full_point = 4
    tool = tools[4]
    id_name = "person_id"
  score = 0
  rated_result = ""
  for id in id_list:
    id_profile = await tool.ainvoke({id_name: id})
    system_prompt = (f"""You will receive two text inputs: a person's resume and their {media_type} profile. \
    Your task is to compare these two sources and identify any discrepancies between them. \
    Examples of such discrepancies include, but are not limited to, \
    inconsistent educational institutions or graduation dates, \
    or conflicting work experience details like employment periods, companies, or job titles.
    You will score the consistency between these two documents. \
    Begin with a total score of {full_point} points. Deduct 1 point for each distinct type of inconsistency found. \
    The minimum possible score is 0.

**Scoring Logic:**
Begin with a total score of {full_point} points. Deduct 1 point for each distinct type of inconsistency found. The minimum possible score is 0.

**INPUT:**
-   **resume (JSON):** {json.dumps(extracted_info, indent=2)}
-   **facebook profile (Text):** {id_profile}

**OUTPUT RULE:**
Analyze the input data for inconsistencies based on the categories above.
Your response must be a JSON object string containing:
-   `discrepancy`: A string listing all identified discrepancies (e.g., "Different cities found. Conflicting work experience details."). If no discrepancies are found, state "No discrepancies found.".
-   `score`: The calculated integer score based on the scoring logic.

**Strict Output Constraints:**
Your response **must and can only** be this JSON object string. You are **strictly forbidden** from including any explanations, conversational text, Markdown formatting (e.g., ` ```json `), or any other extraneous characters.

**JSON Structure Examples:**
{{"discrepancy": "Different cities found. Conflicting work experience details.", "score": 4}}
or
{{"discrepancy": "No discrepancies found.", "score": 6}}
"""
)

    rated_response = llm.invoke(system_prompt)
    result = json.loads(rated_response.content)
    if result["score"]/full_point >= score:
      rated_result = result
      score = result["score"]
      rated_result["score"] = rated_result["score"]/full_point

  return rated_result


In [ ]:
score = await rate_media_info(first_info,"Linkedin",[9,377])
print(score)

{'discrepancy': 'Different cities found. Inconsistent educational institutions or graduation dates. Conflicting work experience details.', 'score': 0.5714285714285714}


In [ ]:
async def generate_score_all_agent(cv:str):
  llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_VERTEX_API_KEY,
    vertexai=False,
    temperature=0
  )
  client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})

  people_score = 0
  info = information_extraction(cv)
  print("==========basic_info==========")
  print(info)
  first_score = first_rating(info)
  print("==========first_rate==========")
  people_score += first_score["score"] * 0.5
  print(f"discrepancy: {first_score["discrepancy"]}\n socre_accumulate: {people_score:.2f}")




  Facebook_possible_id = await seek_for_Facebook_id(extracted_info=info)
  Facebook_rated_result = await rate_media_info(info,"Facebook",Facebook_possible_id)
  print("==========Facebook_rate==========")
  people_score += Facebook_rated_result["score"] * 0.2
  print(f"discrepancy: {Facebook_rated_result["discrepancy"]}\n socre_accumulate: {people_score:.2f}")




  Linkedin_possible_id = await seek_for_linkedin_id(extracted_info=info)
  Linkedin_rated_result = await rate_media_info(info,"Linkedin",Linkedin_possible_id)
  print("==========Linkedin_rate==========")
  people_score += Linkedin_rated_result["score"] * 0.3
  print(f"discrepancy: {Linkedin_rated_result["discrepancy"]}\n socre_accumulate: {people_score:.2f}")




  return people_score


In [ ]:
all_score = [0 for i in range(5)]

In [ ]:
all_score[0] = await generate_score_all_agent(all_cvs[0]["text"])

==========basic_info==========
{'name': 'John Smith', 'headline': 'Marketing Professional', 'city': ['Singapore', 'Kowloon'], 'country': 'Singapore', 'industry': 'Marketing', 'skills': [{'name': 'Content Creation'}, {'name': 'SEO'}, {'name': 'Social Media'}], 'experience': [{'company': 'ByteDance', 'title': 'Engineer', 'start_year': 2020, 'end_year': None, 'is_current': True}], 'education': [{'school': 'McGill University', 'degree': 'BSc', 'field': 'Marketing', 'start_year': None, 'end_year': 2009}]}
==========first_rate==========
discrepancy: Significant disconnect between work experience and listed skills.
 socre_accumulate: 0.38
==========Facebook_rate==========
discrepancy: Different cities found. Conflicting work experience details.
 socre_accumulate: 0.44
==========Linkedin_rate==========
discrepancy: Different cities found. Conflicting work experience details. Inconsistent educational dates.
 socre_accumulate: 0.52


In [ ]:
all_score[1] = await generate_score_all_agent(all_cvs[1]["text"])

==========basic_info==========
{'name': 'Minh Pham', 'headline': 'Design Professional', 'city': ['Beijing', 'Hong Kong'], 'country': 'China', 'industry': 'Design', 'skills': [{'name': 'UI/UX Design'}, {'name': 'Prototyping'}, {'name': 'Graphic Design'}], 'experience': [{'company': 'BCG', 'title': 'Manager', 'start_year': 2022, 'end_year': None, 'is_current': True}, {'company': 'Tencent', 'title': 'Analyst', 'start_year': 2013, 'end_year': 2017, 'is_current': False}], 'education': [{'school': 'The University of Hong Kong', 'degree': 'BSc', 'field': 'Design', 'start_year': None, 'end_year': 2011}]}
==========first_rate==========
discrepancy: No inconsistencies found.
 socre_accumulate: 0.50
==========Facebook_rate==========
discrepancy: Inconsistent educational institutions or degrees found. Conflicting work experience details.
 socre_accumulate: 0.57
==========Linkedin_rate==========
discrepancy: Different cities found. Inconsistent educational institution or graduation dates.
 socre_ac

In [ ]:
all_score[2] = await generate_score_all_agent(all_cvs[2]["text"])

==========basic_info==========
{'name': 'Wei Zhang', 'headline': 'Consulting Professional', 'city': ['Munich', 'Sydney'], 'country': 'Germany', 'industry': 'Consulting', 'skills': [{'name': 'Analytical'}, {'name': 'Data Analysis'}, {'name': 'Problem Solving'}, {'name': 'Business Strategy'}, {'name': 'PowerPoint'}], 'experience': [{'company': 'PwC', 'title': 'Engineer', 'start_year': 2013, 'end_year': None, 'is_current': True}], 'education': [{'school': 'University of Tokyo', 'degree': 'BSc', 'field': 'Consulting', 'start_year': None, 'end_year': 2015}]}
==========first_rate==========
discrepancy: Starting work before latest education graduation.
 socre_accumulate: 0.38
==========Facebook_rate==========
discrepancy: Inconsistent educational institutions.
 socre_accumulate: 0.51
==========Linkedin_rate==========
discrepancy: Different cities found. Inconsistent educational institutions. Conflicting work experience details.
 socre_accumulate: 0.58


In [ ]:
all_score[3] = await generate_score_all_agent(all_cvs[3]["text"])

==========basic_info==========
{'name': 'Rahul Sharma', 'headline': 'Legal Professional', 'city': ['Singapore', 'Philippines'], 'country': None, 'industry': 'Legal', 'skills': [{'name': 'Compliance'}, {'name': 'Litigation'}, {'name': 'Contract Review'}, {'name': 'Web3'}, {'name': 'Machine Learning'}, {'name': 'Quantum Computing'}], 'experience': [{'company': 'Microsoft', 'title': 'Senior Engineer', 'start_year': 2021, 'end_year': 2027, 'is_current': False}, {'company': 'StartupXYZ', 'title': 'Consultant', 'start_year': 2020, 'end_year': 2023, 'is_current': False}], 'education': [{'school': 'Tsinghua University', 'degree': 'PhD', 'field': 'Legal Studies', 'start_year': None, 'end_year': 2021}]}
==========first_rate==========
discrepancy: Overlapping work experience dates, starting work before latest education completion, a significant disconnect between education and listed skills, and a significant disconnect between work experience and listed skills.
 socre_accumulate: 0.00
==========

In [ ]:
all_score[4] = await generate_score_all_agent(all_cvs[4]["text"])

==========basic_info==========
{'name': 'Rahul Sharma', 'headline': 'AI Professional', 'city': ['London', 'Hong Kong', 'Singapore'], 'country': None, 'industry': 'AI', 'skills': [{'name': 'Machine Learning & AI'}, {'name': 'Advanced AI Systems'}, {'name': 'Machine Learning (ML)'}, {'name': 'Natural Language Processing (NLP)'}, {'name': 'TensorFlow'}, {'name': 'PyTorch'}, {'name': 'Python'}], 'experience': [{'company': 'EY', 'title': 'Senior Engineer', 'start_year': None, 'end_year': None, 'is_current': True}, {'company': 'StartupXYZ', 'title': 'Consultant', 'start_year': 2019, 'end_year': 2021, 'is_current': False}, {'company': 'DataForge', 'title': 'Senior Analyst', 'start_year': 2016, 'end_year': None, 'is_current': True}, {'company': 'UrbanFlow', 'title': 'Lead Scientist', 'start_year': 2010, 'end_year': 2017, 'is_current': False}], 'education': [{'school': 'University of Tokyo', 'degree': 'PhD', 'field': 'Artificial Intelligence', 'start_year': None, 'end_year': 2012}]}
==========f

In [ ]:
print(all_score)

[0.5166666666666666, 0.7166666666666667, 0.5833333333333333, 0.075, 0.4583333333333333]


In [ ]:
# This block provides you some tests to get faminilar with our MCP server

# # Test 1: Search Facebook users (exact match)
# await tools[0].ainvoke({'q': "Alex Chan", 'limit': 5})

# # Test 2: Search Facebook users (fuzzy match with typo)
# await tools[0].ainvoke({'q': "Alx Chn", 'limit': 5, 'fuzzy': True})

# # Test 3: Get Facebook profile
# await tools[1].ainvoke({'user_id': 123})

# # Test 4: Get Facebook mutual friends
# await tools[2].ainvoke({'user_id_1': 123, 'user_id_2': 456})

# # Test 5: Search LinkedIn people (exact match)
# await tools[3].ainvoke({'q': "Python", 'location': "Hong Kong", 'limit': 5})

# # Test 6: Search LinkedIn people (fuzzy match with typo)
# await tools[3].ainvoke({'q': "Python", 'location': "Hong Kong", 'limit': 5, 'fuzzy': True})

# # Test 7: Get LinkedIn profile
# await tools[4].ainvoke({'person_id': 456})

# Test 8: Get LinkedIn interactions
await tools[5].ainvoke({'person_id': 456})

CancelledError: 

# Evaluation code

In the test phase, you will be given 5 CV files with fixed names:

    CV_1.pdf, CV_2.pdf, CV_3.pdf, CV_4.pdf, CV_5.pdf

Your system must process these CVs and output a list of 5 scores,
one score per CV, in the same order:

    scores = [s1, s2, s3, s4, s5]

Each score must be a float in the range [0, 1], representing the
reliability or confidence that the CV is valid (or meets the task criteria).

The ground-truth labels are binary:

    groundtruth = [0 or 1, ..., 0 or 1]

Each CV is evaluated independently using a threshold of 0.5:

- If score > 0.5 and groundtruth == 1 → Full credit
- If score ≤ 0.5 and groundtruth == 0 → Full credit
- Otherwise → No credit

In other words, 0.5 is the decision threshold.

- Each CV contributes equally.
- Final score = (number of correct decisions) / 5


In [ ]:
# =====================================================
#  Evaluation code
# =====================================================

def evaluate(scores, groundtruth, threshold=0.5):
    """
    scores: list of floats in [0, 1], length = 5
    groundtruth: list of ints (0 or 1), length = 5
    """
    assert len(scores) == 5
    assert len(groundtruth) == 5

    correct = 0
    decisions = []

    for s, gt in zip(scores, groundtruth):
        pred = 1 if s > threshold else 0
        decisions.append(pred)
        if pred == gt:
            correct += 1

    final_score = correct / len(scores)

    return {
        "decisions": decisions,
        "correct": correct,
        "total": len(scores),
        "final_score": final_score
    }


In [ ]:
scores = all_score # Your code should generate this list [0.2, 0.3, 0.4, 0.5, 0.6]
groundtruth = [1, 1, 1, 0, 0] # Do not modify

result = evaluate(scores, groundtruth)
print(result)


{'decisions': [1, 1, 1, 0, 0], 'correct': 5, 'total': 5, 'final_score': 1.0}
